In [1]:
import pandas as pd
from datasets import load_dataset

print("--- BẮT ĐẦU QUY TRÌNH CHUẨN BỊ VÀ LÀM SẠCH DỮ LIỆU ---")

# 1. Tải dữ liệu gốc từ Hugging Face Hub
hf_dataset = load_dataset("bmd1905/vi-error-correction-2.0")

# 2. Chuyển đổi sang dạng Pandas DataFrame để xử lý lọc trùng
train_df = hf_dataset['train'].to_pandas()
test_df = hf_dataset['test'].to_pandas()

# 3. Tiến hành làm sạch dữ liệu (Data Cleaning)
train_df = train_df.dropna(subset=['input', 'output'])
test_df = test_df.dropna(subset=['input', 'output'])

train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

train_df = train_df[train_df['input'].str.strip() != train_df['output'].str.strip()]
test_df = test_df[test_df['input'].str.strip() != test_df['output'].str.strip()]

# 4. Trích xuất ngẫu nhiên chuẩn 800.000 dòng (Phương án 2 - Kiểm soát 7-8 tiếng train)
print(f"Tổng số mẫu train sạch đang có: {len(train_df)} dòng.")
print("Đang cắt giảm tập dữ liệu xuống 800.000 dòng...")
train_df = train_df.sample(n=800000, random_state=42)

# 5. Xuất thành file CSV lưu tại phân vùng làm việc của Kaggle
train_df.to_csv("/kaggle/working/train_clean.csv", index=False)
test_df.to_csv("/kaggle/working/val_clean.csv", index=False)

print("\n--- HOÀN THÀNH BƯỚC 1 ---")
print(f"Tập Train sạch: {len(train_df)} dòng | Tập Validation sạch: {len(test_df)} dòng.")
print("Hai file 'train_clean.csv' và 'val_clean.csv' đã sẵn sàng trong bộ nhớ làm việc!")

--- BẮT ĐẦU QUY TRÌNH CHUẨN BỊ VÀ LÀM SẠCH DỮ LIỆU ---


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vi.train.csv:   0%|          | 0.00/911M [00:00<?, ?B/s]

vi.test.csv:   0%|          | 0.00/31.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2628492 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/99996 [00:00<?, ? examples/s]

Tổng số mẫu train sạch đang có: 2167847 dòng.
Đang cắt giảm tập dữ liệu xuống 800.000 dòng...



--- HOÀN THÀNH BƯỚC 1 ---
Tập Train sạch: 800000 dòng | Tập Validation sạch: 82369 dòng.
Hai file 'train_clean.csv' và 'val_clean.csv' đã sẵn sàng trong bộ nhớ làm việc!


In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    MBartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

# ===== 1. CẤU HÌNH HỆ THỐNG =====
MODEL_NAME = "vinai/bartpho-syllable"
OUTPUT_DIR = "/kaggle/working/models/bartpho_finetuned"
MAX_LEN = 256

# Đường dẫn file dữ liệu sạch trên Kaggle Environment
TRAIN_PATH = "/kaggle/working/train_clean.csv"
VAL_PATH = "/kaggle/working/val_clean.csv"

# Kiểm tra sự tồn tại của file dữ liệu trước khi chạy để tránh lỗi đọc file trống
if not os.path.exists(TRAIN_PATH) or not os.path.exists(VAL_PATH):
    raise FileNotFoundError("Không tìm thấy file train_clean.csv hoặc val_clean.csv. Hãy chắc chắn bạn đã chạy xong Cell 1!")

# ===== 2. TẢI DỮ LIỆU DẠNG STREAMING (CHỐNG TRÀN RAM) =====
print("Đang cấu hình luồng đọc dữ liệu Streaming từ ổ đĩa...")
train_dataset = load_dataset('csv', data_files=TRAIN_PATH, split='train', streaming=True)
eval_dataset = load_dataset('csv', data_files=VAL_PATH, split='train', streaming=True)

# ===== 3. KHỞI TẠO BỘ MÃ HÓA TỪ VỰNG (TOKENIZER) =====
print(f"Đang tải Tokenizer cho cấu trúc: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(example):
    # Ép kiểu dữ liệu về chuỗi ký tự, xử lý triệt để trường hợp dòng trống (NaN/Null)
    inputs = [str(text) if text is not None else "" for text in example["input"]]
    targets = [str(text) if text is not None else "" for text in example["output"]]
    
    # Mã hóa chuỗi văn bản đầu vào và đầu ra dựa trên cấu hình Max Length
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=MAX_LEN, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Áp dụng hàm ánh xạ Tokenize theo cơ chế nạp từng cụm dữ liệu nhỏ (Batched)
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# ===== 4. KHỞI TẠO KIẾN TRÚC MÔ HÌNH NỀN BARTPHO =====
print("Đang tải kiến trúc mô hình BARTpho từ Hugging Face Hub (Sẽ mất ít phút)...")
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ===== 5. CẤU HÌNH THAM SỐ HUẤN LUYỆN CHỐNG TRÀN VRAM & CHỐNG TREO ĐĨA =====
print("Thiết lập cấu hình tham số huấn luyện nâng cao...")
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    
    # Cấu hình tối ưu bộ nhớ đồ họa đa card (GPU T4 x2)
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=16,     
    max_steps=6250,                     # Chạy trọn vẹn tập dữ liệu 800.000 dòng (6250 steps * 128 batch size)
    
    # CHIẾN THUẬT QUẢN LÝ Ổ ĐĨA AN TOÀN TUYỆT ĐỐI:
    eval_strategy="steps",
    eval_steps=2000,                    # Tăng mốc đánh giá để máy chạy liên tục, giảm độ trễ đồng bộ
    save_steps=2000,                    # Đúng chu kỳ 2000 steps mới ghi file xuống đĩa tạm của Kaggle
    save_total_limit=1,                 # QUAN TRỌNG: Chỉ giữ lại 1 bản checkpoint mới nhất, xóa các bản cũ để chống tràn đĩa
    
    logging_steps=100,                  # Hiển thị tiến trình Loss sau mỗi 100 steps để theo dõi trực quan
    predict_with_generate=False,        # Tắt chế độ tự sinh chữ khi eval để tăng tốc độ huấn luyện lên gấp 3 lần
    load_best_model_at_end=False,       # Tắt chức năng đồng bộ ngược để ngăn chặn lỗi nghẽn I/O cuối luồng chạy
    
    # Kích hoạt phần cứng tăng tốc tính toán song song ma trận
    fp16=True,                          # Chạy Deep Learning chế độ 16-bit tiết kiệm dung lượng VRAM
    dataloader_pin_memory=True,         # Đóng băng vùng nhớ RAM để đẩy dữ liệu sang GPU nhanh hơn
    optim="adamw_torch_fused",          # Sử dụng thuật toán tối ưu hóa gộp nhân đồ họa (Fused Optimizer)
    report_to="none"                    # Ngắt kết nối với các nền tảng log bên ngoài (Wandb, Tensorboard) để tránh lỗi mạng
)

# ===== 6. KHỞI TẠO BỘ PHỐI HỢP TRAINER =====
# Loại bỏ EarlyStoppingCallback để đảm bảo luồng Streaming chạy thông suốt không bị ngắt quãng giữa chừng
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

# ===== 7. KÍCH HOẠT QUÁ TRÌNH FINE-TUNING XUYÊN ĐÊM =====
print("\n--- BẮT ĐẦU KÍCH HOẠT TIẾN TRÌNH HUẤN LUYỆN CHÍNH THỨC ---")
print("Hệ thống sẽ chạy ngầm. Bạn có thể tắt máy tính đi ngủ sau khi bấm Commit thành công.")
trainer.train(resume_from_checkpoint=False)

# ===== 8. XUẤT THÀNH PHẨM TRỌNG SỐ MÔ HÌNH CUỐI CÙNG =====
print("\nĐang đóng gói và kết xuất thư mục trọng số tối ưu cuối cùng...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\n--- HỆ THỐNG ĐÃ HOÀN THÀNH QUÁ TRÌNH HUẤN LUYỆN XUẤT SẮC ---")

Đang cấu hình luồng đọc dữ liệu Streaming từ ổ đĩa...


Đang tải Tokenizer cho cấu trúc: vinai/bartpho-syllable...


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
